---
## Latihan Mandiri

Jalankan ulang cell pembuatan `SparkSession` dan dataset (Sub-bab 12.2) sebelum mengerjakan latihan berikut.

In [7]:
import os

# Menggunakan Java 17
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

# Menghindari konflik instalasi Spark
os.environ.pop("SPARK_HOME", None)

print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))

JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
SPARK_HOME: None


In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Pertemuan11-MLlibRegresi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [9]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 2000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo"]

data = pd.DataFrame({
    "unit_terjual": np.random.randint(1, 15, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "rating": np.random.randint(1, 6, size=n),
})
# Total pendapatan dipengaruhi unit & harga (dengan sedikit variasi acak, seperti data real world)
data["pendapatan"] = data["unit_terjual"] * data["harga_satuan"] + np.random.normal(0, 20000, size=n)

df = spark.createDataFrame(data)
df.show(5)
print("Total baris:", df.count())

+------------+------------+------------+----------+------+------------------+
|unit_terjual|harga_satuan|    kategori|      kota|rating|        pendapatan|
+------------+------------+------------+----------+------+------------------+
|           7|      150000|     Fashion|  Magelang|     4| 997913.0612957058|
|           4|      100000|Rumah Tangga|  Semarang|     4| 389490.8317158621|
|          13|      150000|  Elektronik|Yogyakarta|     4|1936047.6709878189|
|          11|      100000|Rumah Tangga|  Magelang|     5|1080918.9109179082|
|           8|      150000|     Makanan|  Semarang|     2|1191696.4943767258|
+------------+------------+------------+----------+------+------------------+
only showing top 5 rows

Total baris: 2000


In [10]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan11").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.createDataFrame(data)
print("Siap untuk latihan. Total baris:", df.count())

Siap untuk latihan. Total baris: 2000


**Soal 1.** Ulangi proses `StringIndexer` untuk kolom `kategori` dan `kota`, lalu buat `VectorAssembler` **baru** yang HANYA menggunakan tiga fitur: `unit_terjual`, `harga_satuan`, dan `rating` (tanpa `kategori_idx`/`kota_idx`). Beri nama kolom hasil `features_sederhana`.

In [11]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# 1. StringIndexer untuk kategori dan kota
indexer_kategori = StringIndexer(inputCol="kategori", outputCol="kategori_idx")
indexer_kota = StringIndexer(inputCol="kota", outputCol="kota_idx")

df_indexed = indexer_kategori.fit(df).transform(df)
df_indexed = indexer_kota.fit(df_indexed).transform(df_indexed)

# 2. VectorAssembler hanya untuk 3 fitur (unit_terjual, harga_satuan, rating)
assembler_sederhana = VectorAssembler(
    inputCols=["unit_terjual", "harga_satuan", "rating"],
    outputCol="features_sederhana"
)

df_latihan = assembler_sederhana.transform(df_indexed)
df_latihan.select("features_sederhana", "pendapatan").show(5, truncate=False)

+-------------------+------------------+
|features_sederhana |pendapatan        |
+-------------------+------------------+
|[7.0,150000.0,4.0] |997913.0612957058 |
|[4.0,100000.0,4.0] |389490.8317158621 |
|[13.0,150000.0,4.0]|1936047.6709878189|
|[11.0,100000.0,5.0]|1080918.9109179082|
|[8.0,150000.0,2.0] |1191696.4943767258|
+-------------------+------------------+
only showing top 5 rows



**Soal 2.** Latih model `LinearRegression` baru menggunakan `features_sederhana` dari Soal 1 (gunakan train/test split yang sama, `seed=42`). Bandingkan nilai **R²** model ini dengan model utama pada Sub-bab 11.4 yang menggunakan 5 fitur — apakah performanya lebih baik, sama, atau lebih buruk?

In [12]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Split data dengan seed=42 yang sama
train_sederhana, test_sederhana = df_latihan.randomSplit([0.8, 0.2], seed=42)

# Melatih model menggunakan features_sederhana
lr_sederhana = LinearRegression(featuresCol="features_sederhana", labelCol="pendapatan")
model_sederhana = lr_sederhana.fit(train_sederhana)

# Evaluasi pada data testing
pred_sederhana = model_sederhana.transform(test_sederhana)
evaluator_r2 = RegressionEvaluator(labelCol="pendapatan", predictionCol="prediction", metricName="r2")
r2_sederhana = evaluator_r2.evaluate(pred_sederhana)

print(f"R² Model Sederhana (3 fitur) : {r2_sederhana:.4f}")
print(f"R² Model Utama (5 fitur)     : 0.8796")

R² Model Sederhana (3 fitur) : 0.8801
R² Model Utama (5 fitur)     : 0.8796


**Soal 3.** Menggunakan model utama (5 fitur) dari Sub-bab 11.4, tampilkan **5 baris prediksi dengan selisih (`pendapatan - prediction`) terbesar** — yaitu prediksi yang paling meleset. (Petunjuk: tambahkan kolom baru berisi nilai absolut selisihnya, lalu urutkan.)

In [13]:
from pyspark.sql.functions import abs as spark_abs, col

# Menggabungkan kembali fitur utama (5 fitur) agar dapat dievaluasi dengan model utama
assembler_utama = VectorAssembler(
    inputCols=["unit_terjual", "harga_satuan", "kategori_idx", "kota_idx", "rating"],
    outputCol="features"
)
df_utama = assembler_utama.transform(df_indexed)
train_utama, test_utama = df_utama.randomSplit([0.8, 0.2], seed=42)

model_utama = LinearRegression(featuresCol="features", labelCol="pendapatan").fit(train_utama)
predictions_utama = model_utama.transform(test_utama)

# Menambahkan kolom selisih absolut dan mengurutkan dari yang terbesar
df_selisih = predictions_utama.withColumn(
    "selisih_absolut", spark_abs(col("pendapatan") - col("prediction"))
)

df_selisih.select(
    "unit_terjual", "harga_satuan", "kategori", "kota", "pendapatan", "prediction", "selisih_absolut"
).orderBy(col("selisih_absolut").desc()).show(5)

+------------+------------+----------+----------+------------------+------------------+------------------+
|unit_terjual|harga_satuan|  kategori|      kota|        pendapatan|        prediction|   selisih_absolut|
+------------+------------+----------+----------+------------------+------------------+------------------+
|           1|      150000|   Fashion|  Semarang|129325.30266583862| 627142.2205709203|497816.91790508165|
|           1|      150000|   Makanan|Yogyakarta|120877.75807042063| 605747.0283431914| 484869.2702727707|
|           1|      150000|Elektronik|Yogyakarta|144667.73973440926| 613784.7449136553| 469117.0051792461|
|          14|      150000|   Makanan|  Magelang| 2120244.488469077|1653177.4600901871|467067.02837888966|
|          14|      150000|Elektronik|Yogyakarta|  2116917.02368938| 1652646.104973831|464270.91871554893|
+------------+------------+----------+----------+------------------+------------------+------------------+
only showing top 5 rows



**Soal 4 (Refleksi singkat).** Dalam 2-3 kalimat: mengapa Spark MLlib mewajibkan kita menggabungkan seluruh fitur menjadi satu kolom `Vector` (melalui `VectorAssembler`), bukan menerima banyak kolom terpisah seperti kebiasaan di pandas/scikit-learn? Tulis jawaban pada markdown cell di bawah ini.

Spark MLlib mewajibkan penggabungan seluruh fitur ke dalam satu kolom bertipe Vector karena arsitektur pemrosesan terdistribusinya dirancang untuk meminimalkan overhead serialisasi dan transfer data antar-node cluster. Dengan merepresentasikan baris fitur sebagai satu objek vektor matematis tunggal (yang mendukung format dense maupun sparse), algoritma MLlib dapat melakukan operasi aljabar linear terdistribusi secara jauh lebih efisien dibandingkan membaca puluhan kolom terpisah.

In [14]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.


---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 12 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7, 9, dan 10) ingin memiliki alat bantu **prediksi omzet harian** tiap cabang, agar dapat merencanakan stok bahan baku dengan lebih baik. Anda ditugaskan membangun model regresi untuk memprediksi `total_penjualan` sebuah pesanan berdasarkan atribut-atributnya.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset.

In [15]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


### Instruksi Pengerjaan

Buat notebook baru **`Tugas11_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, baca `tugas11_penjualan.csv`, lalu kerjakan bagian **A sampai E** berikut:

---

**A. Feature Engineering** *(bobot 20%)*

Gunakan `StringIndexer` untuk mengubah kolom `nama_menu`, `cabang`, dan `waktu_pesan` menjadi indeks numerik. Gabungkan seluruh fitur (`qty`, `harga_satuan`, dan ketiga kolom hasil indexing) menjadi satu kolom `features` menggunakan `VectorAssembler`.

**B. Train/Test Split & Pelatihan Model** *(bobot 20%)*

Bagi data menjadi 75% training dan 25% testing (`seed=101`). Latih model `LinearRegression` dengan `labelCol="total_penjualan"`.

**C. Evaluasi Model** *(bobot 20%)*

Hitung dan tampilkan **RMSE** dan **R²** dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah model ini cukup baik untuk dipakai memprediksi omzet?

**D. Eksperimen Perbandingan Fitur** *(bobot 25%)*

Latih **model kedua** yang HANYA menggunakan fitur `qty` dan `harga_satuan` (tanpa informasi menu/cabang/waktu). Bandingkan R² model kedua ini dengan model utama pada bagian C. Tulis kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi prediksi secara berarti?

**E. Prediksi Skenario Baru** *(bobot 15%)*

Buat sebuah DataFrame baru berisi **3 skenario pesanan (data dummy)** (boleh bebas kombinasi menu/cabang/waktu/qty/harga, namun harus melalui tahapan `StringIndexer` dan `VectorAssembler` yang sama seperti data training — perhatikan: gunakan model `StringIndexer` yang **sama** yang sudah di-`fit()` sebelumnya, jangan membuat `StringIndexer` baru). Tampilkan hasil prediksi `total_penjualan` untuk ketiga skenario tersebut.

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas11_[NPM]_[Nama Lengkap].ipynb` melalui asisten praktikum, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Seluruh cell wajib sudah dijalankan (**Run All**) sebelum dikumpulkan.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Feature Engineering | `StringIndexer` & `VectorAssembler` diterapkan dengan benar | 20% |
| B. Split & Pelatihan | Split proporsional benar, model berhasil dilatih | 20% |
| C. Evaluasi | RMSE & R² dihitung benar, interpretasi masuk akal | 20% |
| D. Eksperimen Perbandingan | Model kedua benar, perbandingan & kesimpulan logis | 25% |
| E. Prediksi Skenario Baru | Ketiga skenario diproses & diprediksi dengan benar | 15% |


In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
spark = SparkSession.builder \
    .appName("Pertemuan12-MLlibKlasifikasi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [17]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder \
    .appName("Tugas11-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# Membaca dataset CSV yang telah dibuat
df_kopi = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
df_kopi.show(5)

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows



In [18]:
# Inisialisasi StringIndexer
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

# Simpan model hasil fit() agar dapat digunakan kembali pada Bagian E
model_idx_menu = idx_menu.fit(df_kopi)
model_idx_cabang = idx_cabang.fit(df_kopi)
model_idx_waktu = idx_waktu.fit(df_kopi)

# Transformasi data
df_kopi_indexed = model_idx_menu.transform(df_kopi)
df_kopi_indexed = model_idx_cabang.transform(df_kopi_indexed)
df_kopi_indexed = model_idx_waktu.transform(df_kopi_indexed)

# Menggabungkan 5 fitur menjadi satu kolom 'features'
assembler_utama = VectorAssembler(
    inputCols=["qty", "harga_satuan", "menu_idx", "cabang_idx", "waktu_idx"],
    outputCol="features"
)

df_kopi_final = assembler_utama.transform(df_kopi_indexed)
df_kopi_final.select("features", "total_penjualan").show(5, truncate=False)

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+-------------------------+------------------+
only showing top 5 rows



In [19]:
# Split 75% training, 25% testing dengan seed=101
train_data, test_data = df_kopi_final.randomSplit([0.75, 0.25], seed=101)

print(f"Jumlah data training : {train_data.count()}")
print(f"Jumlah data testing  : {test_data.count()}")

# Melatih model LinearRegression
lr_utama = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model_utama = lr_utama.fit(train_data)

print("Koefisien:", model_utama.coefficients)
print("Intercept:", model_utama.intercept)

Jumlah data training : 1839
Jumlah data testing  : 661
Koefisien: [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept: -66903.17766485889


In [20]:
prediksi_utama = model_utama.transform(test_data)

eval_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
eval_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse_utama = eval_rmse.evaluate(prediksi_utama)
r2_utama = eval_r2.evaluate(prediksi_utama)

print(f"RMSE Model Utama : Rp {rmse_utama:,.2f}")
print(f"R² Model Utama   : {r2_utama:.4f}")

RMSE Model Utama : Rp 10,900.80
R² Model Utama   : 0.9107


In [21]:
# VectorAssembler kedua hanya menggunakan qty dan harga_satuan
assembler_2fitur = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_2"
)

df_kopi_2fitur = assembler_2fitur.transform(df_kopi)
train_2, test_2 = df_kopi_2fitur.randomSplit([0.75, 0.25], seed=101)

# Latih model kedua
lr_2fitur = LinearRegression(featuresCol="features_2", labelCol="total_penjualan")
model_2fitur = lr_2fitur.fit(train_2)

# Evaluasi R² model kedua
prediksi_2fitur = model_2fitur.transform(test_2)
r2_2fitur = eval_r2.evaluate(prediksi_2fitur)

print(f"R² Model Utama (5 fitur) : {r2_utama:.4f}")
print(f"R² Model Kedua (2 fitur) : {r2_2fitur:.4f}")

R² Model Utama (5 fitur) : 0.9107
R² Model Kedua (2 fitur) : 0.9108


In [22]:
# 1. Membuat DataFrame berisi 3 skenario pesanan dummy
data_dummy = [
    ("Kopi Susu", "Magelang", "Pagi", 3, 20000),
    ("Roti Bakar", "Yogyakarta", "Malam", 2, 25000),
    ("Americano", "Semarang", "Siang", 5, 18000)
]
kolom_dummy = ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"]

df_dummy = spark.createDataFrame(data_dummy, schema=kolom_dummy)

# 2. Transformasi menggunakan model StringIndexer yang SUDAH di-fit pada Bagian A
df_dummy_indexed = model_idx_menu.transform(df_dummy)
df_dummy_indexed = model_idx_cabang.transform(df_dummy_indexed)
df_dummy_indexed = model_idx_waktu.transform(df_dummy_indexed)

# 3. Menggabungkan fitur dengan VectorAssembler yang sama
df_dummy_final = assembler_utama.transform(df_dummy_indexed)

# 4. Prediksi menggunakan model_utama dari Bagian B
hasil_prediksi_dummy = model_utama.transform(df_dummy_final)
hasil_prediksi_dummy.select(
    "nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction"
).show(truncate=False)

+----------+----------+-----------+---+------------+------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+----------+----------+-----------+---+------------+------------------+
|Kopi Susu |Magelang  |Pagi       |3  |20000       |59572.01525544525 |
|Roti Bakar|Yogyakarta|Malam      |2  |25000       |52889.236877203075|
|Americano |Semarang  |Siang      |5  |18000       |98980.56095704187 |
+----------+----------+-----------+---+------------+------------------+



In [23]:
# Menutup SparkSession setelah seluruh bagian tugas selesai
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
